# **AI ASSIGNMENT # 02**
## **I242533**
## **Dawar Ahmed** 
## **DS - *B***

In [1]:
import random

#Card Class
class Card:
    def __init__(self, color, value):
        self.color = color
        self.value = str(value)

    def __repr__(self):
        return f"{self.color} {self.value}"
    
    #Check if two cards match for playing
    def matches(self, other_card):
        return self.color == other_card.color or self.value == other_card.value

#Deck Generator
def generate_deck():
    colors = ['Red', 'Blue', 'Green', 'Yellow']
    numbers = list(range(10))
    
    deck = []
    for color in colors:
        for num in numbers:
            deck.append(Card(color, num))
        #Adding Skip cards
        deck.append(Card(color, 'Skip'))
        
    random.shuffle(deck)
    return deck

#Legal Move Generator
def get_valid_moves(hand, top_card):
    valid_moves = []
    for card in hand:
        if card.matches(top_card):
            valid_moves.append(card)
    return valid_moves

#Testing Step 1
deck = generate_deck()
player1_hand = [deck.pop() for _ in range(5)]
top_card = Card('Red', '5')

print("Top Card on Table:", top_card)
print("Player 1 Hand:", player1_hand)
print("Valid Moves for Player 1:", get_valid_moves(player1_hand, top_card))

Top Card on Table: Red 5
Player 1 Hand: [Red 9, Red 3, Red 2, Blue Skip, Yellow 0]
Valid Moves for Player 1: [Red 9, Red 3, Red 2]


In [2]:
#GameState class
class GameState:
    def __init__(self, ai_cards, opponent1_card_count, opponent2_card_count, top_card, deck_size):
        self.ai_cards = ai_cards
        self.opponent1_cards = opponent1_card_count
        self.opponent2_cards = opponent2_card_count
        self.top_card = top_card
        self.deck = deck_size
        
    def __repr__(self):
        return f"Top: {self.top_card} | AI Cards: {len(self.ai_cards)} | Opp1: {self.opponent1_cards} | Opp2: {self.opponent2_cards}"

#Evaluation Function
def evaluate_state(state, strategy="Defensive"):
    #Formula variables
    c_ai = len(state.ai_cards)
    c_opp = (state.opponent1_cards + state.opponent2_cards) / 2.0
    s_count = sum(1 for card in state.ai_cards if card.value == 'Skip')
    
    #Base Score
    score = 50 - (5 * c_ai) + (2 * c_opp) + (3 * s_count)
    
    #Strategy Tuning
    if strategy == "Defensive":
        if state.opponent1_cards <= 1 or state.opponent2_cards <= 1:
            score -= 20
        score += (2 * s_count)
        
    elif strategy == "Offensive":
        if c_ai <= 1:
            score += 20
        score -= (2 * c_ai)
        
    return score

#Testing Step 2
fake_ai_hand = [Card('Red', '5'), Card('Blue', 'Skip'), Card('Green', '9')]
fake_state = GameState(fake_ai_hand, 2, 4, Card('Red', '3'), 30)

print(fake_state)
print("Defensive AI Score:", evaluate_state(fake_state, "Defensive"))
print("Offensive AI Score:", evaluate_state(fake_state, "Offensive"))

Top: Red 3 | AI Cards: 3 | Opp1: 2 | Opp2: 4
Defensive AI Score: 46.0
Offensive AI Score: 38.0


In [20]:
#Simulate Move Function
def simulate_move(state, move, player_turn):
    new_ai_cards = state.ai_cards.copy()
    new_opponent1_cards = state.opponent1_cards
    new_opponent2_cards = state.opponent2_cards
    new_top_card = state.top_card
    new_deck_size = state.deck
    
    #AI Turn
    if player_turn == 0:
        if move == "Draw":
            if new_deck_size > 0:
                new_ai_cards.append(Card('Unknown', 'Unknown')) 
                new_deck_size -= 1
        else:
            for c in new_ai_cards:
                if c.color == move.color and c.value == move.value:
                    new_ai_cards.remove(c)
                    new_top_card = move
                    break
                    
    #Opponent 1 Turn
    elif player_turn == 1:
        if move == "Draw":
            if new_deck_size > 0:
                new_opponent1_cards += 1
                new_deck_size -= 1
        else:
            new_opponent1_cards -= 1
            new_top_card = move
            
    #Opponent 2 Turn
    elif player_turn == 2:
        if move == "Draw":
            if new_deck_size > 0:
                new_opponent2_cards += 1
                new_deck_size -= 1
        else:
            new_opponent2_cards -= 1
            new_top_card = move
            
    return GameState(new_ai_cards, new_opponent1_cards, new_opponent2_cards, new_top_card, new_deck_size)

#Testing Step 3
test_state = GameState([Card('Red', '5'), Card('Blue', 'Skip')], 3, 2, Card('Red', '3'), 25)
print("Test 1 AI plays card:", simulate_move(test_state, Card('Red', '5'), 0))
print("Test 2 AI draws card:", simulate_move(test_state, "Draw", 0))



Test 1 AI plays card: Top: Red 5 | AI Cards: 1 | Opp1: 3 | Opp2: 2
Test 2 AI draws card: Top: Red 3 | AI Cards: 3 | Opp1: 3 | Opp2: 2


In [21]:
#Minimax Algorithm (Defensive AI)
def minimax(state, depth, alpha, beta, player_turn):
    #Base case
    if depth == 0 or len(state.ai_cards) == 0 or state.opponent1_cards == 0 or state.opponent2_cards == 0:
        return evaluate_state(state, "Defensive"), None
        
    #AI Turn (Maximizing Player)
    if player_turn == 0:
        max_eval = float('-inf')
        best_move = None
        
        valid_moves = get_valid_moves(state.ai_cards, state.top_card)
        if not valid_moves:
            valid_moves = ["Draw"]
            
        for move in valid_moves:
            new_state = simulate_move(state, move, player_turn)
            eval_score, _ = minimax(new_state, depth - 1, alpha, beta, 1)
            
            if eval_score > max_eval:
                max_eval = eval_score
                best_move = move
                
            alpha = max(alpha, eval_score)
            if beta <= alpha:
                break 
        return max_eval, best_move
        
    #Opponents Turn (Minimizing Players)
    else:
        min_eval = float('inf')
        possible_moves = [state.top_card, "Draw"]
        
        for move in possible_moves:
            new_state = simulate_move(state, move, player_turn)
            next_turn = 2 if player_turn == 1 else 0
            
            eval_score, _ = minimax(new_state, depth - 1, alpha, beta, next_turn)
            
            if eval_score < min_eval:
                min_eval = eval_score
                
            beta = min(beta, eval_score)
            if beta <= alpha:
                break 
        return min_eval, None

#Testing Step 4
best_score, best_move = minimax(test_state, 3, float('-inf'), float('inf'), 0)
print("Minimax Best Move:", best_move)
print("Expected Score:", best_score)

Minimax Best Move: Red 5
Expected Score: 33.0
